In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.gold.uptime_daily AS
SELECT
  vendor,
  component_id,
  component_name,
  DATE(observed_at) AS date,
  COUNT(*) AS total_polls,
  SUM(CASE WHEN status = 'operational' THEN 1 ELSE 0 END) AS operational_polls,
  SUM(CASE WHEN status = 'under_maintenance' THEN 1 ELSE 0 END) AS maintenance_polls,
  SUM(CASE WHEN status NOT IN ('operational', 'under_maintenance') THEN 1 ELSE 0 END) AS degraded_or_down_polls,
  ROUND(100.0 * SUM(CASE WHEN status = 'operational' THEN 1 ELSE 0 END) / COUNT(*), 2) AS uptime_pct,
  ROUND(
    100.0 * SUM(CASE WHEN status = 'operational' THEN 1 ELSE 0 END)
    / NULLIF(SUM(CASE WHEN status != 'under_maintenance' THEN 1 ELSE 0 END), 0),
    2
  ) AS uptime_pct_excl_maintenance
FROM vendor_status.silver.components
WHERE COALESCE(is_group, false) = false
GROUP BY vendor, component_id, component_name, DATE(observed_at);

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.gold.incident_summary AS
WITH incident_timeline AS (
  SELECT
    vendor,
    source_type,
    incident_id,
    MIN(created_at) AS started_at,
    MAX(CASE WHEN status = 'resolved' THEN created_at END) AS resolved_at
  FROM vendor_status.silver.incident_updates
  GROUP BY vendor, source_type, incident_id
)
SELECT
  i.vendor,
  i.source_type,
  i.incident_id,
  i.name,
  i.impact,
  i.status AS current_status,
  t.started_at,
  t.resolved_at,
  CASE
    WHEN t.resolved_at IS NOT NULL
    THEN ROUND((unix_timestamp(CAST(t.resolved_at AS TIMESTAMP)) - unix_timestamp(CAST(t.started_at AS TIMESTAMP))) / 60.0, 1)
    ELSE ROUND((unix_timestamp(current_timestamp()) - unix_timestamp(CAST(t.started_at AS TIMESTAMP))) / 60.0, 1)
  END AS elapsed_minutes,
  t.resolved_at IS NULL AS is_ongoing
FROM vendor_status.silver.incidents i
JOIN incident_timeline t
  ON i.vendor = t.vendor AND i.source_type = t.source_type AND i.incident_id = t.incident_id;

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.gold.vendor_incident_metrics AS
SELECT
  vendor,
  COUNT(*) AS total_incidents,
  SUM(CASE WHEN is_ongoing THEN 1 ELSE 0 END) AS ongoing_incidents,
  ROUND(AVG(CASE WHEN is_ongoing = false THEN elapsed_minutes END), 1) AS avg_mttr_minutes
FROM vendor_status.gold.incident_summary
WHERE source_type = 'incident'
GROUP BY vendor;

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.gold.vendor_freshness AS
SELECT
  vendor,
  MAX(observed_at) AS last_observed_at,
  ROUND(
    (unix_timestamp(current_timestamp()) - unix_timestamp(CAST(MAX(observed_at) AS TIMESTAMP))) / 60.0,
    1
  ) AS minutes_since_last_poll
FROM vendor_status.silver.page_status
GROUP BY vendor;

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.gold.vendor_current_status AS
SELECT vendor, indicator, description, observed_at
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY vendor ORDER BY observed_at DESC) AS rn
  FROM vendor_status.silver.page_status
) WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.gold.component_status_distribution AS
SELECT vendor, status, COUNT(*) AS count
FROM vendor_status.silver.components
WHERE is_group = false
GROUP BY vendor, status;

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.gold.top_affected_components AS
SELECT vendor, component_id, component_name, COUNT(*) AS change_count
FROM vendor_status.silver.component_changes
GROUP BY vendor, component_id, component_name
ORDER BY change_count DESC
LIMIT 50;

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.gold.dim_vendor AS
SELECT DISTINCT vendor
FROM vendor_status.silver.page_status;

In [0]:
%sql
CREATE OR REPLACE TABLE vendor_status.gold.dim_component AS
SELECT DISTINCT vendor, component_id, component_name
FROM vendor_status.silver.components
WHERE is_group = false;

In [0]:
%sql
SELECT * FROM vendor_status.gold.uptime_daily ORDER BY date DESC LIMIT 10;

In [0]:
%sql
SELECT * FROM vendor_status.gold.incident_summary;

In [0]:
%sql
SELECT * FROM vendor_status.gold.vendor_incident_metrics;

In [0]:
%sql
SELECT * FROM vendor_status.gold.vendor_freshness;

In [0]:
%sql
SELECT vendor, component_id, component_name, status, observed_at
FROM vendor_status.silver.components
WHERE component_id IN ('171wlzvlxttk', 'xjfk3l4dykrs')
ORDER BY observed_at DESC
LIMIT 10;

In [0]:
%sql
SELECT vendor, indicator FROM vendor_status.gold.vendor_current_status;

In [0]:
%sql
SELECT * FROM vendor_status.gold.uptime_daily WHERE vendor = 'OpenAI';

In [0]:
%sql
SELECT component_name, is_group, status, COUNT(*) AS times_seen
FROM vendor_status.silver.components
WHERE vendor = 'OpenAI'
GROUP BY component_name, is_group, status
ORDER BY is_group, component_name;

In [0]:
%sql
SELECT explode(components) AS c
FROM vendor_status.bronze.status_checks
WHERE page.name = 'OpenAI'
ORDER BY _bronze_loaded_at DESC
LIMIT 1

In [0]:
%sql
SELECT vendor, name, source_type, elapsed_minutes, is_ongoing
FROM vendor_status.gold.incident_summary
ORDER BY elapsed_minutes DESC;